In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# reading FACE July data from directory, this is already the CO2 data every 3 hours
July = pd.read_csv('../July.csv', index_col=0)
July = July.iloc[::3]
July.reset_index(inplace=True, drop=True)
July

,station,valid,tmpc,relh,sped,tmpK,index,YEAR,DOY,HOUR,RING,CO2,WIND,WIND_DIR,SOLAR_ANG
0,OQT,2008-07-01 00:53:00,22.78,42.48,3.45,295.93,8395,2008,182,0,0,496.1,1.38,260,-30.77
1,OQT,2008-07-01 03:53:00,16.67,69.51,0.00,289.82,8410,2008,182,3,0,479.6,1.18,220,-18.63
2,OQT,2008-07-01 06:53:00,13.89,83.21,0.00,287.04,8425,2008,182,6,0,488.6,1.13,260,11.32
3,OQT,2008-07-01 09:53:00,12.22,89.95,0.00,285.37,8440,2008,182,9,0,375.6,2.01,240,47.09
4,OQT,2008-07-01 12:53:00,17.78,72.53,0.00,290.93,8455,2008,182,12,0,369.5,1.99,220,76.47
5,OQT,2008-07-01 15:53:00,25.56,37.38,5.75,298.71,8470,2008,182,15,0,365.8,3.32,260,51.45
6,OQT,2008-07-01 18:53:00,27.78,32.84,0.00,300.93,8485,2008,182,18,0,366.1,1.41,320,15.35
7,OQT,2008-07-01 21:53:00,28.89,26.55,5.75,302.04,8500,2008,182,21,0,429.2,0.35,240,-15.69
8,OQT,2008-07-02 00:53:00,25.00,38.74,0.00,298.15,8515,2008,183,0,0,483.6,0.37,0,-30.83
9,OQT,2008-07-02 03:53:00,18.33,70.28,0.00,291.48,8530,2008,183,3,0,525.5,0.29,260,-18.71


In [4]:
# Here we add rate constants kf1 and kf2 as well as water fraction to the dataset using functions below
def interpolate(RH_in):
    kf1 = [1.37, 1.48, 1.726, 1.35, 1.78]
    kf2 = [1.71, 12.67,  33.39, 2.06, 14.54]
    T = [298, 299, 298, 308, 308]
    vol = [0.002, 0.01, 0.02, 0.01, 0.02]
    RH_list = []

    for i in range(len(T)):
        P_H2O_sat = np.exp(77.34 - 7235/T[i]-8.2*np.log(T[i]) + 0.005711*T[i])
        P_H2O = 1e5*vol[i]
        RH_list.append(P_H2O/P_H2O_sat*100)

    RH_kf = np.array([RH_list, kf1, kf2]).T
    RH_kf = RH_kf[RH_kf[:, 0].argsort()]

    kf1_hat = np.interp(RH_in, RH_kf[:, 0], RH_kf[:, 1])
    kf2_hat = np.interp(RH_in, RH_kf[:, 0], RH_kf[:, 2])

    return kf1_hat, kf2_hat

def H2O_vol(T_g, RH, P):
    P_H2O_sat = 0.61078*np.exp(17.269*(T_g-273.15)/(T_g-273.15+237.29))*1000
    P_H2O = P_H2O_sat*RH
    vol = P_H2O/P
    return vol

After adding all inputs to the dataset, we are going to import functions that will be used to run gPROMS in Python

In [5]:
import subprocess
import os
import time
import sys
import logging
from threading import Timer
import psutil
import matplotlib.pyplot as plt
%matplotlib inline

In [6]:
import pandas as pd


def gPLOT2df(filename):
    """
    Convert .gPLOT file to a pandas dataframe

    Args:
        filename (str) : Name of the gPLOT file.

    Returns:
        df (DataFrame): Pandas dataframe with the time indexes on the rows and the variables on the columns.
    """
    with open(filename, 'r') as f:
        # Read the first line with the number of variables
        nvar = int(f.readline())
        # Read the variablenames
        vars = [f.readline().strip() for i in range(0, nvar)]
        # Read the variablevalues at each time point
        df = pd.read_csv(f, header=None, index_col=False)
        ntime = int((len(df) - 1) / (nvar + 1))
        # Reshape and drop termination element
        df = pd.DataFrame(data=df.values[0:-1].reshape((ntime, nvar + 1)), columns=['Time'] + vars)
    return df

In [7]:
def call_gORUN(gORUN_path, gENCRYPT_name, activity, process, password, working_dir='./', time_out=24*3600,
               verbose=True):
    """
    Calls gORUN and waits for the execution to complete.

    Args:
        gORUN (str): Path to gORUN.bat (must end with gORUN.bat)
        gENCRYPT_name (str): Name of the gENCRYPT file to execute
        activity (str): Activity type  ('sim'|'opt'|'est')
        process (str): Name of the process in the gPROMS file
        password (str): Encryption password of the process in the gPROMS file
        timeout (float): Time out in seconds. After this time the gO:RUN process will be terminated.
        working_dir (str): If the function is not called from the directory containing the input directory,
            use this to change to the right working directory first. The working directory will be set back to the
            original one before terminating.

    Returns:
        result (logical): Returns True for succes, False for error
    """
    # Start timer
    start_time = time.time()

    mpicmd_part_1 = "mpiexec.exe -n 1 -localonly"
    mpicmd_part_2 = "-mpmd"
    mpicmd_part_3 = ": -n NW gCluster.exe"

    # Construct gORUN command
    #gORUN_cmd = [gORUN_path, gENCRYPT_name, activity, process, password]
    gORUN_options =  gENCRYPT_name + " " +  activity + " " +  process + " " +  password
    gORUN_output = ' > ' + activity + '_output.txt'

    np=1
    if np > 1:
        gORUN_path = gORUN_path.rstrip('.bat') + '.exe'
        gORUN_cmd = mpicmd_part_1 + " \"" + gORUN_path + "\" " + mpicmd_part_2 + " " + gORUN_options + " " + \
                    mpicmd_part_3
    else:
        gORUN_cmd = gORUN_path + " " + gORUN_options + gORUN_output
    # Run gO:RUN and log output to file
#    try:
#        with open('./pygORUN.log', 'w') as f:
#     f = open('test.log', 'wb')

    #print(gORUN_cmd)
    print('gPROMS executed')
    process = subprocess.Popen(gORUN_cmd, cwd=working_dir,
                                       encoding='ascii')
#            for line in iter(process.stdout.readline, ""):
#                if verbose:
#                    sys.stdout.write(line)
#                f.write(line.rstrip('\n'))
#                f.flush()
#   except subprocess.CalledProcessError as e:
#        logging.error('Error calling gO:RUN')

    my_timer = Timer(time_out, lambda: kill_process(process))

    try:
        my_timer.start()
        output, error = process.communicate()
    finally:
        my_timer.cancel()

    elapsed_time = time.time() - start_time
    if elapsed_time > time_out:
        print("Activity timed out")


    if process.returncode == 0 and elapsed_time < time_out:
        return True
    else:
        return False

def kill_process(process_to_kill,timestamp="zzz"):
    # ----------------------------------------------------------------------------------------------
    # FEP new solution on killing gORUNs that hang
    try:
        parent = psutil.Process(process_to_kill.pid)
        children = parent.children(recursive=True)
        for child in children:
            child.kill()
        psutil.wait_procs(children, timeout=1)

        # having cleaned up all the children also clean up self
        process_to_kill.terminate()

    except:
        print("Unable to kill")
        pass

       

In [9]:
# we add one more input, which is the cooling water temperature of adsorption step
h2o_temp = np.zeros(July.shape[0])

for i in range(July.shape[0]):
    if July['tmpK'].iloc[i] <= 278:
        # we dont want the cooling water to be either too cold or too hot
        h2o_temp[i] = 278
    elif July['tmpK'].iloc[i] >= 303:
        h2o_temp[i] = 303
    else:
        h2o_temp[i] = July['tmpK'].iloc[i]

July['h2o_temperature'] = h2o_temp

kf1 = np.zeros(July.shape[0])
kf2 = np.zeros(July.shape[0])

for i in range(July.shape[0]):
    kf1[i], kf2[i] = interpolate([July['relh'].iloc[i]])

July['kf1, bar-1 s-1 (mol/kg)-1'] = kf1
July['kf2, bar2 s-1'] = kf2

# We assume the atmospheric pressure is 96890 Pa using 2008 July data
July['pressure'] = 96890

July['H2O_vol'] = H2O_vol(July['tmpK'], July['relh']*0.01, July['pressure'])
July


,station,valid,tmpc,relh,sped,tmpK,index,YEAR,DOY,HOUR,RING,CO2,WIND,WIND_DIR,SOLAR_ANG,h2o_temperature,"kf1, bar-1 s-1 (mol/kg)-1","kf2, bar2 s-1",pressure,H2O_vol
0,OQT,2008-07-01 00:53:00,22.78,42.48,3.45,295.93,8395,2008,182,0,0,496.1,1.38,260,-30.77,295.93,1.767537,18.890409,96890,0.012154
1,OQT,2008-07-01 03:53:00,16.67,69.51,0.00,289.82,8410,2008,182,3,0,479.6,1.18,220,-18.63,289.82,1.726000,33.390000,96890,0.013613
2,OQT,2008-07-01 06:53:00,13.89,83.21,0.00,287.04,8425,2008,182,6,0,488.6,1.13,260,11.32,287.04,1.726000,33.390000,96890,0.013631
3,OQT,2008-07-01 09:53:00,12.22,89.95,0.00,285.37,8440,2008,182,9,0,375.6,2.01,240,47.09,285.37,1.726000,33.390000,96890,0.013210
4,OQT,2008-07-01 12:53:00,17.78,72.53,0.00,290.93,8455,2008,182,12,0,369.5,1.99,220,76.47,290.93,1.726000,33.390000,96890,0.015237
5,OQT,2008-07-01 15:53:00,25.56,37.38,5.75,298.71,8470,2008,182,15,0,365.8,3.32,260,51.45,298.71,1.777381,15.454184,96890,0.012634
6,OQT,2008-07-01 18:53:00,27.78,32.84,0.00,300.93,8485,2008,182,18,0,366.1,1.41,320,15.35,300.93,1.617310,13.525897,96890,0.012648
7,OQT,2008-07-01 21:53:00,28.89,26.55,5.75,302.04,8500,2008,182,21,0,429.2,0.35,240,-15.69,302.04,1.441419,9.521196,96890,0.010906
8,OQT,2008-07-02 00:53:00,25.00,38.74,0.00,298.15,8515,2008,183,0,0,483.6,0.37,0,-30.83,298.15,1.774756,16.370511,96890,0.012665
9,OQT,2008-07-02 03:53:00,18.33,70.28,0.00,291.48,8530,2008,183,3,0,525.5,0.29,260,-18.71,291.48,1.726000,33.390000,96890,0.015284


Now we have a full dataset ready for gPROMS and optimizations

In [1]:
from PyDDSBB import DDSBB, DDSBBModel

The block below is used to do the DDSBB optimization

In [11]:
t_opt = np.zeros([July.shape[0], 3])

# Type here how many data points you want to optimize in one batch
i_batch = 20
for i in np.arange(i_batch):
    sample = July.iloc[i]

    # Write in the txt file for gRPOMS input
    def write_txt(cycle_time, No_Cycle, Ads0_time, input_file = 'inputs.txt'):
        # input kf1 ,kf2 
        parameter_sens = [sample['kf1, bar-1 s-1 (mol/kg)-1']*1e-5, sample['kf2, bar2 s-1']*1e-10]
        # input all other parameters
        geo_inputs = [sample['H2O_vol'], sample['CO2']*1e-6, sample['tmpK'], sample['pressure'], sample['h2o_temperature']]
        f = open(input_file, "w+")
        f.write("Ads0_time"+"\n")
        f.write(str(np.round(Ads0_time))+"\n"+"\n")     

        f.write("Ads_time"+"\n")
        f.write(str(np.round(cycle_time[0]))+"\n"+"\n")

        f.write("Des_time"+"\n")
        f.write(str(np.round(cycle_time[1]))+"\n"+"\n")

        f.write("No_Cycle"+"\n")
        f.write(str(No_Cycle)+"\n"+"\n")

        f.write("kf_1"+"\n")
        f.write(str(parameter_sens[0])+"\n"+"\n")

        f.write("kf_2"+"\n")
        f.write(str(parameter_sens[1])+"\n"+"\n")

        f.write("y_H2O_0"+"\n")
        f.write(str(geo_inputs[0])+"\n"+"\n")

        f.write("y_CO2_0"+"\n")
        f.write(str(geo_inputs[1])+"\n"+"\n")

        f.write("T_feed"+"\n")
        f.write(str(geo_inputs[2])+"\n"+"\n")

        f.write("P_atm"+"\n")
        f.write(str(geo_inputs[3])+"\n"+"\n")

        f.write("T_h2o"+"\n")
        f.write(str(geo_inputs[4])+"\n"+"\n")

        #print(cycle_time)
        f.close()

    Dict = {}

    # call gPROMS and run
    def sim_output(cycle_time):

        ads = np.round(cycle_time[0])
        Ads0 = ads*2
        vac1 = 5
        vac2 = 200
        vac3 = 100
        des = np.round(cycle_time[1])
        press1 = 20
        press2 = 200
        press3 = 5

        cycle = np.ceil(15000/(cycle_time[0]+cycle_time[1]))

        print(cycle_time, cycle)
        write_txt(cycle_time, No_Cycle = cycle, Ads0_time = Ads0, input_file = 'inputs.txt')
        # make sure this path is correct for your gPROMS directory!!
        gORUN_path = 'C:/Program Files/PSE/gPROMS-core_2021.2.0.55235/bin/goRUN.bat'
        # Type in the name of your .ENCRYPT file
        gENCRYPT_name = 'H2O_export'
        activity = 'sim'
        process = 'H2O_export'
        password = 'H2O_export'

        call_gORUN(gORUN_path, gENCRYPT_name, activity, process, password, working_dir='./', time_out=24*3600,
                    verbose=True)
        # Directory of your gPROMS simulation output
        filename = 'C:/Users/xcai65/OneDrive - Georgia Institute of Technology/study/research/NETL/script/Shubham/optimize_cost/output/H2O_export.gPLOT'
        output_results = gPLOT2df(filename)

        report_Des = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)-ads-press3-press2-press1
        report_Ads = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)
        report_Vac3 = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)-ads-press3-press2-press1-des-vac3

        try:

            prod_mass = output_results['Channel.PROD_MASS'][output_results['Time'] == report_Des].iloc[0] #mol/kg fiber/hr

            Out_CO2_des = output_results['Channel.OUT_CO2_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
            Out_N2_des = output_results['Channel.OUT_N2_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
            Out_H2O_des = output_results['Channel.OUT_H2O_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
            In_CO2_ads = output_results['Channel.IN_CO2_ads'][output_results['Time'] == report_Ads].iloc[0] #mol/channel

            Recovery = Out_CO2_des/In_CO2_ads
            Purity_CO2 = Out_CO2_des/(Out_CO2_des+Out_N2_des)

            E_blower = output_results['Channel.E_blower_ash'][output_results['Time'] == report_Ads].iloc[0]/(Out_CO2_des*44.02*1e-6) # kWh/tCO2
            E_vac = output_results['Channel.E_vac'][output_results['Time'] == report_Des].iloc[0]/(Out_CO2_des*44.02*1e-6) # kWh/tCO2
            Q_sens_ads = -(output_results['Channel.H_ads_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                        output_results['Channel.H_ads_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
            Q_sens_CO2 = (output_results['Channel.H_CO2_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                        output_results['Channel.H_CO2_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
            Q_sens_H2O = (output_results['Channel.H_H2O_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                        output_results['Channel.H_H2O_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2    
            Q_ads_H2O = 50700*Out_H2O_des*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
            Q_ads_CO2 = (output_results['Channel.H_ads_CO2'][output_results['Time'] == report_Vac3].iloc[0]-\
                        output_results['Channel.H_ads_CO2'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2  
            
            E_blower = E_blower*27.78
            E_vac = E_vac*(Out_CO2_des+Out_N2_des)/(Out_CO2_des+Out_N2_des+Out_H2O_des)
            total_cost = (E_blower+E_vac)*0.05+(Q_sens_ads+Q_sens_CO2+Q_ads_CO2+Q_sens_H2O+Q_ads_H2O)*0.015

        except:
            prod_mass = 0
            Recovery = 0
            Out_CO2_des = 0
            Out_N2_des = 0
            Out_H2O_des = 0
            #Purity_H2O = 0
            E_blower = 0
            E_vac = 0
            Q_sens_ads = 0
            Q_sens_CO2 = 0
            Q_sens_H2O = 0
            Q_ads_H2O = 0
            Q_ads_CO2 = 0
            total_cost = 250
        


        print(prod_mass, total_cost)


        return total_cost

    # DDSBB optimization step
    model = DDSBBModel.Problem()
    model.add_objective(sim_output, sense = 'minimize')

    model.add_variable(300., 3600.)
    model.add_variable(300., 3600.)

    model_solver = DDSBB(20, split_method = 'equal_bisection', \
                            variable_selection = 'longest_side', multifidelity = 'nn', \
                            stop_option = {'absolute_tolerance': 0.05, 'relative_tolerance': 0.05, \
                                            'minimum_bound': 0.05, 'sampling_limit': 300, 'time_limit': 10000})


    model_solver.optimize(model)
    t_opt[i, 0] = model_solver.get_optimum()
    t_opt[i, 1:3] = model_solver.get_optimizer()

    print('Optimized cycle time for Month {} is {}'.format(i, t_opt[i, :]))

[300. 300.] 25.0
gPROMS executed
1.201404481364426 115.81915448577216
[3600. 3600.] 3.0
gPROMS executed
0 250
[ 596.29730317 3159.29550999] 4.0
gPROMS executed
0.3448930438062184 90.97101035894735
[3114.35497227  788.03786023] 4.0
gPROMS executed
0.5455027773330583 110.17337303376362
[1320.80870627 1735.83118625] 5.0
gPROMS executed
0.6160904309064269 84.3003338443512
[2258.02755255 2916.46133831] 3.0
gPROMS executed
0.4024632527360302 96.07671060973135
[3259.61378089 2112.02631051] 3.0
gPROMS executed
0.4020046951349224 111.84087290883488
[1909.80564328  740.60069205] 6.0
gPROMS executed
0.7575192288865314 91.59397007790048
[ 408.91416216 1352.35091193] 9.0
gPROMS executed
0.5661683511367939 103.94455376730534
[ 857.95658804 2415.08804078] 5.0
gPROMS executed
0.4878678987120121 84.25766679326657
[1536.30145058 2601.84933947] 4.0
gPROMS executed
0.4723383959311662 86.34081595486987
[1256.83933441  525.96131055] 9.0
gPROMS executed
0.9824481187607742 85.87509891507237
[1232.97503924 322

RuntimeError: ERROR: Failed to train DDCU

In [25]:
print(t_opt)

[[  83.25051787 1076.79862204 2387.66621783]
 [ 100.2296519  1448.71292553 1346.0682509 ]
 [ 116.20978145 1373.2295116  1804.18191783]
 [ 138.60206982 1681.93603146 2461.06941159]
 [ 114.59246284 1721.27339912 1488.61689627]
 [  95.62265192 1274.4225767  3567.29138734]
 [  97.71267101 1198.28900574 1204.32235546]
 [  90.78085018  991.45225838  906.87645171]
 [  84.08655177 1044.23829831 1544.97118565]
 [  98.89975325 1320.37242339 2487.79642452]
 [ 109.54722496 1369.94445816 1645.24054414]
 [ 129.4526497  1590.23526102 1481.35598828]
 [ 119.40381278 1743.75       1537.5       ]
 [ 101.31255596 1203.45833781 2112.20920222]
 [  99.77422876  962.73113899  992.59091467]
 [  92.89241981  942.70842561 1326.82507268]
 [  87.24552644  994.08373359 1039.24667811]
 [  95.31579565 1324.50896562 2511.92228785]
 [ 110.93902867 1281.19881833 2074.90021548]
 [ 128.73501095 1561.25055783 1575.98767073]
 [   0.            0.            0.        ]
 [   0.            0.            0.        ]
 [   0.   

In [26]:
df_out = pd.DataFrame(t_opt, columns=['E_cost', 'Ads_t, s', 'Des_t, s'])

July = July.reset_index(drop=True)
July = pd.concat([July, df_out], axis = 1)
July


,station,valid,tmpc,relh,sped,tmpK,index,YEAR,DOY,HOUR,...,"kf1, bar-1 s-1 (mol/kg)-1","kf2, bar2 s-1",pressure,H2O_vol,E_cost,"Ads_t, s","Des_t, s",E_cost,"Ads_t, s","Des_t, s"
0,OQT,2008-07-01 00:53:00,22.78,42.48,3.45,295.93,8395,2008,182,0,...,1.767537,18.890409,96890,0.012154,64.051434,1417.627221,1491.831213,83.250518,1076.798622,2387.666218
1,OQT,2008-07-01 03:53:00,16.67,69.51,0.00,289.82,8410,2008,182,3,...,1.726000,33.390000,96890,0.013613,79.386534,1988.200884,1981.100348,100.229652,1448.712926,1346.068251
2,OQT,2008-07-01 06:53:00,13.89,83.21,0.00,287.04,8425,2008,182,6,...,1.726000,33.390000,96890,0.013631,93.426403,1852.994753,2196.511280,116.209781,1373.229512,1804.181918
3,OQT,2008-07-01 09:53:00,12.22,89.95,0.00,285.37,8440,2008,182,9,...,1.726000,33.390000,96890,0.013210,112.496731,2222.901529,2165.653455,138.602070,1681.936031,2461.069412
4,OQT,2008-07-01 12:53:00,17.78,72.53,0.00,290.93,8455,2008,182,12,...,1.726000,33.390000,96890,0.015237,91.596659,2362.500000,1950.000000,114.592463,1721.273399,1488.616896
5,OQT,2008-07-01 15:53:00,25.56,37.38,5.75,298.71,8470,2008,182,15,...,1.777381,15.454184,96890,0.012634,73.528730,1506.692712,1460.430979,95.622652,1274.422577,3567.291387
6,OQT,2008-07-01 18:53:00,27.78,32.84,0.00,300.93,8485,2008,182,18,...,1.617310,13.525897,96890,0.012648,74.743761,1433.704797,1371.310072,97.712671,1198.289006,1204.322355
7,OQT,2008-07-01 21:53:00,28.89,26.55,5.75,302.04,8500,2008,182,21,...,1.441419,9.521196,96890,0.010906,68.433496,1162.000733,1176.408103,90.780850,991.452258,906.876452
8,OQT,2008-07-02 00:53:00,25.00,38.74,0.00,298.15,8515,2008,183,0,...,1.774756,16.370511,96890,0.012665,64.666658,1389.233331,1818.726070,84.086552,1044.238298,1544.971186
9,OQT,2008-07-02 03:53:00,18.33,70.28,0.00,291.48,8530,2008,183,3,...,1.726000,33.390000,96890,0.015284,78.409278,1829.748308,1808.089060,98.899753,1320.372423,2487.796425


In [12]:
def write_txt(parameter_sens, geo_inputs, cycle_time, No_Cycle, Ads0_time, input_file = 'inputs.txt'):
    # x are the independent input, K_sb is the adjustable
    f = open(input_file, "w+")
    f.write("Ads0_time"+"\n")
    f.write(str(np.round(Ads0_time))+"\n"+"\n")     

    f.write("Ads_time"+"\n")
    f.write(str(np.round(cycle_time[0]))+"\n"+"\n")

    f.write("Des_time"+"\n")
    f.write(str(np.round(cycle_time[1]))+"\n"+"\n")

    f.write("No_Cycle"+"\n")
    f.write(str(No_Cycle)+"\n"+"\n")

    f.write("kf_1"+"\n")
    f.write(str(parameter_sens[0])+"\n"+"\n")

    f.write("kf_2"+"\n")
    f.write(str(parameter_sens[1])+"\n"+"\n")

    f.write("y_H2O_0"+"\n")
    f.write(str(geo_inputs[0])+"\n"+"\n")

    f.write("y_CO2_0"+"\n")
    f.write(str(geo_inputs[1])+"\n"+"\n")

    f.write("T_feed"+"\n")
    f.write(str(geo_inputs[2])+"\n"+"\n")

    f.write("P_atm"+"\n")
    f.write(str(geo_inputs[3])+"\n"+"\n")

    f.write("T_h2o"+"\n")
    f.write(str(geo_inputs[4])+"\n"+"\n")

    #print(cycle_time)
    f.close()
    
def sim_output(parameter_sens, geo_inputs, cycle_time):

    ads = np.round(cycle_time[0])
    Ads0 = ads*2
    vac1 = 5
    vac2 = 200
    vac3 = 100
    des = np.round(cycle_time[1])
    press1 = 20
    press2 = 200
    press3 = 5

    cycle = np.ceil(15000/(cycle_time[0]+cycle_time[1]))

    print(cycle_time, cycle)
    write_txt(parameter_sens, geo_inputs, cycle_time, No_Cycle = cycle, Ads0_time = Ads0, input_file = 'inputs.txt')


    gORUN_path = 'C:/Program Files/PSE/gPROMS-core_2021.2.0.55235/bin/goRUN.bat'
    gENCRYPT_name = 'H2O_export'
    activity = 'sim'
    process = 'H2O_export'
    password = 'H2O_export'

    call_gORUN(gORUN_path, gENCRYPT_name, activity, process, password, working_dir='./', time_out=24*3600,
                verbose=True)

    
    filename = 'C:/Users/xcai65/OneDrive - Georgia Institute of Technology/study/research/NETL/script/FACE/cost/output/H2O_export.gPLOT'
    output_results = gPLOT2df(filename)

    report_Des = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)-ads-press3-press2-press1
    report_Ads = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)
    report_Vac3 = Ads0+vac1+vac2+vac3+des+press1+press2+press3+ads+(vac1+vac2+vac3+des+press1+press2+press3+ads)*(cycle-1)-ads-press3-press2-press1-des-vac3

    try:
        #output_results['Time'] = np.round(output_results['Time'].values)
        prod_mass = output_results['Channel.PROD_MASS'][output_results['Time'] == report_Des].iloc[0] #mol/kg fiber/hr

        Out_CO2_des = output_results['Channel.OUT_CO2_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
        Out_N2_des = output_results['Channel.OUT_N2_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
        Out_H2O_des = output_results['Channel.OUT_H2O_des'][output_results['Time'] == report_Des].iloc[0] #mol/channel
        In_CO2_ads = output_results['Channel.IN_CO2_ads'][output_results['Time'] == report_Ads].iloc[0] #mol/channel

        Recovery = Out_CO2_des/In_CO2_ads
        Purity_CO2 = Out_CO2_des/(Out_CO2_des+Out_N2_des)
        #Purity_H2O = Out_H2O_des/(Out_CO2_des+Out_N2_des+Out_H2O_des)

        E_blower = output_results['Channel.E_blower_ash'][output_results['Time'] == report_Ads].iloc[0]/(Out_CO2_des*44.02*1e-6) # kWh/tCO2
        E_vac = output_results['Channel.E_vac'][output_results['Time'] == report_Des].iloc[0]/(Out_CO2_des*44.02*1e-6) # kWh/tCO2
        Q_sens_ads = -(output_results['Channel.H_ads_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                    output_results['Channel.H_ads_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
        Q_sens_CO2 = (output_results['Channel.H_CO2_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                    output_results['Channel.H_CO2_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
        Q_sens_H2O = (output_results['Channel.H_H2O_sens'][output_results['Time'] == report_Vac3].iloc[0]-\
                    output_results['Channel.H_H2O_sens'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2    
        Q_ads_H2O = 50700*Out_H2O_des*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2
        Q_ads_CO2 = (output_results['Channel.H_ads_CO2'][output_results['Time'] == report_Vac3].iloc[0]-\
                    output_results['Channel.H_ads_CO2'][output_results['Time'] == report_Des].iloc[0])*1e-3/(Out_CO2_des*44.02*1e-6)/3600 # kWh/tCO2  

        E_blower = E_blower*27.78
        E_vac = E_vac*(Out_CO2_des+Out_N2_des)/(Out_CO2_des+Out_N2_des+Out_H2O_des)
    except:
        prod_mass = 0
        Recovery = 0
        Out_CO2_des = 0
        Out_N2_des = 0
        Out_H2O_des = 0
        #Purity_H2O = 0
        E_blower = 0
        E_vac = 0
        Q_sens_ads = 0
        Q_sens_CO2 = 0
        Q_sens_H2O = 0
        Q_ads_H2O = 0
        Q_ads_CO2 = 0


    
    return ([E_blower, E_vac, Q_sens_ads, Q_sens_CO2, Q_sens_H2O, Q_ads_CO2, Q_ads_H2O, prod_mass])


In [21]:
July = pd.read_csv('July_cost_20.csv', index_col=0)

In [22]:
July.columns

Index(['station', 'valid', 'tmpc', 'relh', 'sped', 'tmpK', 'index', 'YEAR',
       'DOY', 'HOUR', 'RING', 'CO2', 'WIND', 'WIND_DIR', 'SOLAR_ANG',
       'h2o_temperature', 'kf1, bar-1 s-1 (mol/kg)-1', 'kf2, bar2 s-1',
       'pressure', 'H2O_vol', 'E_cost', 'Ads_t, s', 'Des_t, s'],
      dtype='object')

In [23]:
output_E = np.zeros([July.shape[0], 8])

for i in range(July.shape[0]):
    sample = July.iloc[i, :]
    parameter_sens = [sample['kf1, bar-1 s-1 (mol/kg)-1']*1e-5, sample['kf2, bar2 s-1']*1e-10]
    geoinputs = [sample['H2O_vol'], sample['CO2']*1e-6, sample['tmpK'], sample['pressure'], sample['h2o_temperature']]
    cycle_time = [sample['Ads_t, s'], sample['Des_t, s']]

    output_E[i, :]= sim_output(parameter_sens, geoinputs, cycle_time)
    print(output_E[i, :],  i)

[1076.798622, 2387.666218] 5.0
gPROMS executed
[4.87326390e+02 1.20282180e+02 1.57280645e+03 6.53404047e+01
 3.53725580e+02 6.36766209e+02 8.96167883e+02 5.08168907e-01] 0
[1448.712926, 1346.068251] 6.0
gPROMS executed
[5.15098712e+02 1.16449237e+02 1.33251455e+03 6.48101895e+01
 7.39951512e+02 6.44267632e+02 1.79515916e+03 8.01932024e-01] 1
[1373.229512, 1804.181918] 5.0
gPROMS executed
[4.82534755e+02 1.15108738e+02 1.36190011e+03 6.41397668e+01
 1.07881142e+03 6.42897011e+02 2.60739045e+03 7.13873257e-01] 2
[1681.936031, 2461.069412] 4.0
gPROMS executed
[6.06526992e+02 1.14443057e+02 1.42458567e+03 6.36071907e+01
 1.37731863e+03 6.41807142e+02 3.32958556e+03 5.33551155e-01] 3
[1721.273399, 1488.616896] 5.0
gPROMS executed
[6.42601272e+02 1.16051363e+02 1.38167731e+03 6.50049349e+01
 8.86980648e+02 6.45890723e+02 2.13110198e+03 6.64749077e-01] 4
[1274.422577, 3567.291387] 4.0
gPROMS executed
[6.83210534e+02 1.21392833e+02 1.79701276e+03 6.53358272e+01
 3.34544979e+02 6.36025901e+02 8

KeyboardInterrupt: 

In [ ]:
df_output_E = pd.DataFrame(output_E, columns = ['E_blower', 'E_vac', 'Q_sens_ads', 'Q_sens_CO2', 'Q_sens_H2O', 'Q_ads_CO2', 'Q_ads_H2O', 'Optimized Prod'])
df_output_E['Cost_E, $/tCO2'] = df_output_E.iloc[:, :2].sum(axis = 1)*0.05
df_output_E['Cost_H, $/tCO2'] = df_output_E.iloc[:, 2:7].sum(axis = 1)*0.015

df_final = pd.concat([July, df_output_E], axis = 1)
df_final.to_csv('July_cost_20.csv')